# Model Results & Evaluation

Covers: training curves · per-subclass recall · PR/ROC curves · confusion matrices · per-original-label metrics · cross-model comparison · Phase 7 diagnosis.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report,
    matthews_corrcoef,
)

sys.path.insert(0, str(Path("..") / "src"))
from kvasir_triage.data.labels import SUSPICIOUS_CLASSES, NORMAL_CLASSES, BINARY_LABEL

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

In [ ]:
RUNS_ROOT = Path("../outputs/runs")

CONFIG_NAMES = ["densenet161_pretrained", "densenet161_scratch", "efficientnet_b0"]
DISPLAY_NAMES = {
    "densenet161_pretrained": "DenseNet-161 pretrained",
    "densenet161_scratch":    "DenseNet-161 scratch",
    "efficientnet_b0":        "EfficientNet-B0",
}
COLORS = {
    "densenet161_pretrained": "#2196F3",
    "densenet161_scratch":    "#FF5722",
    "efficientnet_b0":        "#4CAF50",
}
FOLD_LS = {0: "-", 1: "--"}
THRESHOLD = 0.5

SUSPICIOUS = sorted(SUSPICIOUS_CLASSES)
NORMAL     = sorted(NORMAL_CLASSES)
ALL_SUBCLASSES = SUSPICIOUS + NORMAL

In [ ]:
def find_runs(config_name: str) -> dict:
    """Return {fold: run_dir} for the most recent run of each fold."""
    result = {}
    for fold in [0, 1]:
        candidates = sorted(RUNS_ROOT.glob(f"{config_name}_fold{fold}_*"))
        if candidates:
            result[fold] = candidates[-1]
    return result


def load_metrics(run_dir: Path) -> pd.DataFrame:
    path = run_dir / "metrics.jsonl"
    if not path.exists():
        return pd.DataFrame()
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    # Replace JSON null with NaN
    return df.replace({None: float("nan")})


def load_preds(run_dir: Path):
    """Return (logits, labels, subclasses) arrays or None."""
    path = run_dir / "val_preds_best.npz"
    if not path.exists():
        return None
    d = np.load(path, allow_pickle=True)
    return d["logits"], d["labels"], d["subclasses"].astype(str)


# ── discover ──────────────────────────────────────────────────────────────────
all_runs = {cfg: find_runs(cfg) for cfg in CONFIG_NAMES}

print("Discovered runs:")
for cfg, folds in all_runs.items():
    if folds:
        for fold, path in folds.items():
            preds_ok = (path / "val_preds_best.npz").exists()
            print(f"  {cfg} fold {fold}  →  {path.name}  [preds: {'yes' if preds_ok else 'no'}]")
    else:
        print(f"  {cfg}: no runs found")

## 1 · Training curves

In [ ]:
fig, axes = plt.subplots(2, len(CONFIG_NAMES), figsize=(5 * len(CONFIG_NAMES), 8), sharey="row")

for col, cfg in enumerate(CONFIG_NAMES):
    ax_loss = axes[0, col]
    ax_prauc = axes[1, col]
    color = COLORS[cfg]
    has_any = False

    for fold, run_dir in all_runs[cfg].items():
        df = load_metrics(run_dir)
        if df.empty:
            continue
        has_any = True
        ls = FOLD_LS[fold]
        label = f"fold {fold}"

        if "train_loss" in df.columns:
            ax_loss.plot(df["epoch"], df["train_loss"], color=color, ls=ls, label=label)
            # mark loss function switch
            if "loss_switched" in df.columns:
                switch_epochs = df[df["loss_switched"] == True]["epoch"]
                for ep in switch_epochs:
                    ax_loss.axvline(ep, color="orange", lw=1, ls=":", alpha=0.8)

        if "val_pr_auc" in df.columns:
            ax_prauc.plot(df["epoch"], df["val_pr_auc"], color=color, ls=ls, label=label)

    title = DISPLAY_NAMES[cfg]
    ax_loss.set_title(title, fontsize=9)
    ax_loss.set_xlabel("Epoch")
    ax_prauc.set_xlabel("Epoch")
    if col == 0:
        ax_loss.set_ylabel("Train loss")
        ax_prauc.set_ylabel("Val PR-AUC")
    if has_any:
        ax_loss.legend(fontsize=8)
        ax_prauc.legend(fontsize=8)

    ax_loss.grid(True, alpha=0.3)
    ax_prauc.grid(True, alpha=0.3)
    ax_prauc.set_ylim(0, 1)

# legend for loss-switch marker
switch_patch = Line2D([0], [0], color="orange", ls=":", label="BCE→Focal switch")
fig.legend(handles=[switch_patch], loc="upper right", fontsize=8)

fig.suptitle("Training curves", fontsize=12, y=1.01)
fig.tight_layout()
plt.show()

## 2 · Final metrics summary

In [ ]:
rows = []
for cfg in CONFIG_NAMES:
    for fold, run_dir in all_runs[cfg].items():
        df = load_metrics(run_dir)
        if df.empty:
            continue
        # use best val_pr_auc epoch
        best = df.loc[df["val_pr_auc"].idxmax()]
        rows.append({
            "model":      DISPLAY_NAMES[cfg],
            "fold":       fold,
            "epoch":      int(best["epoch"]),
            "pr_auc":     round(float(best["val_pr_auc"]), 4),
            "mcc":        round(float(best.get("val_mcc", float("nan"))), 4),
            "npv":        round(float(best.get("val_npv",  float("nan"))), 4),
            "loss_fn":    best.get("loss_fn", "bce"),
        })

summary_df = pd.DataFrame(rows)

if not summary_df.empty:
    # CV mean ± std per model
    cv = (
        summary_df.groupby("model")[["pr_auc", "mcc", "npv"]]
        .agg(["mean", "std"])
        .round(4)
    )
    print("Per-fold results:")
    display(summary_df)
    print("\nCV mean ± std:")
    display(cv)
else:
    print("No completed runs found.")

In [ ]:
if not summary_df.empty:
    cv_mean = summary_df.groupby("model")["pr_auc"].mean().sort_values(ascending=False)
    cv_std  = summary_df.groupby("model")["pr_auc"].std().reindex(cv_mean.index)

    fig, ax = plt.subplots(figsize=(7, 4))
    colors = [COLORS[k] for k in CONFIG_NAMES if DISPLAY_NAMES[k] in cv_mean.index]
    bars = ax.bar(cv_mean.index, cv_mean.values, yerr=cv_std.values,
                  color=colors, capsize=5, width=0.5, error_kw={"elinewidth": 1.5})
    ax.set_ylabel("Val PR-AUC (CV mean ± std)")
    ax.set_ylim(0, 1.05)
    ax.axhline(0.85, color="red", ls="--", lw=1, label="Phase-7 target (0.85)")
    ax.legend(fontsize=8)
    for bar, val in zip(bars, cv_mean.values):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f"{val:.3f}",
                ha="center", va="bottom", fontsize=9)
    ax.set_title("Cross-validated PR-AUC by model")
    fig.tight_layout()
    plt.show()

## 3 · Per-subclass recall over training

In [ ]:
for cfg in CONFIG_NAMES:
    folds_data = {}
    recall_cols = None
    for fold, run_dir in all_runs[cfg].items():
        df = load_metrics(run_dir)
        if df.empty:
            continue
        cols = [c for c in df.columns if c.startswith("recall_")]
        if not cols:
            continue
        recall_cols = cols
        folds_data[fold] = df

    if not folds_data or not recall_cols:
        continue

    subclasses = [c.replace("recall_", "") for c in recall_cols]
    n = len(subclasses)
    ncols = 4
    nrows = (n + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 2.8), sharey=True)
    axes_flat = axes.flatten() if n > 1 else [axes]

    for i, (sub, col) in enumerate(zip(subclasses, recall_cols)):
        ax = axes_flat[i]
        for fold, df in folds_data.items():
            if col in df.columns:
                ax.plot(df["epoch"], df[col], ls=FOLD_LS[fold],
                        color=COLORS[cfg], label=f"fold {fold}")
        ax.axhline(0.85, color="red", ls=":", lw=1)
        ax.set_title(sub, fontsize=8)
        ax.set_ylim(0, 1.05)
        ax.set_xlabel("Epoch", fontsize=7)
        ax.set_ylabel("Recall", fontsize=7)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(f"{DISPLAY_NAMES[cfg]} — per-subclass recall", fontsize=11)
    fig.tight_layout()
    plt.show()

## 4 · Final per-subclass recall heatmap

In [ ]:
heatmap_rows = []
for cfg in CONFIG_NAMES:
    for fold, run_dir in all_runs[cfg].items():
        df = load_metrics(run_dir)
        if df.empty:
            continue
        best = df.loc[df["val_pr_auc"].idxmax()]
        row = {"model": DISPLAY_NAMES[cfg], "fold": fold}
        for col in [c for c in df.columns if c.startswith("recall_")]:
            row[col.replace("recall_", "")] = float(best[col])
        heatmap_rows.append(row)

if heatmap_rows:
    hm_df = pd.DataFrame(heatmap_rows).set_index(["model", "fold"])
    subclass_cols = [c for c in hm_df.columns]

    fig, ax = plt.subplots(figsize=(max(8, len(subclass_cols) * 1.4), max(3, len(hm_df) * 0.6)))
    im = ax.imshow(hm_df[subclass_cols].values.astype(float), aspect="auto",
                   vmin=0, vmax=1, cmap="RdYlGn")
    ax.set_xticks(range(len(subclass_cols)))
    ax.set_xticklabels(subclass_cols, rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(len(hm_df)))
    ax.set_yticklabels([f"{m}\nfold {f}" for m, f in hm_df.index], fontsize=8)
    for i in range(len(hm_df)):
        for j in range(len(subclass_cols)):
            v = hm_df.iloc[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=7, color="black" if 0.25 < v < 0.75 else "white")
    plt.colorbar(im, ax=ax, label="Recall")
    ax.set_title("Per-subclass recall at best epoch (suspicious classes only)")
    fig.tight_layout()
    plt.show()
else:
    print("No per-subclass recall data available yet.")

## 5 · PR curves & ROC curves

In [ ]:
fig, (ax_pr, ax_roc) = plt.subplots(1, 2, figsize=(12, 5))

has_any = False
for cfg in CONFIG_NAMES:
    for fold, run_dir in all_runs[cfg].items():
        preds = load_preds(run_dir)
        if preds is None:
            continue
        logits, labels, _ = preds
        probs = 1 / (1 + np.exp(-logits))
        has_any = True
        label_str = f"{DISPLAY_NAMES[cfg]} f{fold}"
        color = COLORS[cfg]

        # PR curve
        prec, rec, _ = precision_recall_curve(labels, probs)
        ap = average_precision_score(labels, probs)
        ax_pr.plot(rec, prec, color=color, ls=FOLD_LS[fold],
                   label=f"{label_str} (AP={ap:.3f})")

        # ROC curve
        fpr, tpr, _ = roc_curve(labels, probs)
        roc_auc = auc(fpr, tpr)
        ax_roc.plot(fpr, tpr, color=color, ls=FOLD_LS[fold],
                    label=f"{label_str} (AUC={roc_auc:.3f})")

# baselines
ax_pr.axhline(labels.mean() if has_any else 0.5, color="grey", ls=":", lw=1, label="Chance")
ax_roc.plot([0, 1], [0, 1], color="grey", ls=":", lw=1, label="Chance")

ax_pr.set_xlabel("Recall"); ax_pr.set_ylabel("Precision")
ax_pr.set_xlim(0, 1); ax_pr.set_ylim(0, 1.05)
ax_pr.set_title("Precision-Recall curves")
ax_pr.legend(fontsize=7, loc="lower left")
ax_pr.grid(True, alpha=0.3)

ax_roc.set_xlabel("FPR"); ax_roc.set_ylabel("TPR")
ax_roc.set_xlim(0, 1); ax_roc.set_ylim(0, 1.05)
ax_roc.set_title("ROC curves")
ax_roc.legend(fontsize=7, loc="lower right")
ax_roc.grid(True, alpha=0.3)

if not has_any:
    for ax in (ax_pr, ax_roc):
        ax.text(0.5, 0.5, "val_preds_best.npz\nnot found",
                ha="center", va="center", transform=ax.transAxes, fontsize=10, color="grey")

fig.tight_layout()
plt.show()

## 6 · Confusion matrices

In [ ]:
cfgs_with_preds = [
    cfg for cfg in CONFIG_NAMES
    if any((run_dir / "val_preds_best.npz").exists() for run_dir in all_runs[cfg].values())
]

if not cfgs_with_preds:
    print("No val_preds_best.npz files found yet.")
else:
    fig, axes = plt.subplots(1, len(cfgs_with_preds), figsize=(4.5 * len(cfgs_with_preds), 4))
    if len(cfgs_with_preds) == 1:
        axes = [axes]

    for ax, cfg in zip(axes, cfgs_with_preds):
        # aggregate both folds
        all_logits, all_labels = [], []
        for fold, run_dir in all_runs[cfg].items():
            preds = load_preds(run_dir)
            if preds is None:
                continue
            logits, labels, _ = preds
            all_logits.append(logits)
            all_labels.append(labels)

        if not all_logits:
            continue

        logits = np.concatenate(all_logits)
        labels = np.concatenate(all_labels)
        probs  = 1 / (1 + np.exp(-logits))
        preds_bin = (probs >= THRESHOLD).astype(int)

        cm = confusion_matrix(labels, preds_bin)
        cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)

        im = ax.imshow(cm_pct, vmin=0, vmax=1, cmap="Blues")
        tick_labels = ["Normal", "Suspicious"]
        ax.set_xticks([0, 1]); ax.set_xticklabels(tick_labels)
        ax.set_yticks([0, 1]); ax.set_yticklabels(tick_labels)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")

        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{cm[i,j]}\n({cm_pct[i,j]:.1%})",
                        ha="center", va="center",
                        color="white" if cm_pct[i, j] > 0.5 else "black", fontsize=9)

        ax.set_title(DISPLAY_NAMES[cfg], fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046)

    fig.suptitle("Confusion matrices (both folds combined, threshold=0.5)", fontsize=11)
    fig.tight_layout()
    plt.show()

## 7 · Per-original-label metrics

Precision, recall, and F1 for each of the 11 subclasses (binary predictions at threshold 0.5).

In [ ]:
def per_label_metrics(logits, labels, subclasses, threshold=0.5):
    """Compute precision, recall, F1, support for each subclass."""
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= threshold).astype(int)
    rows = []
    for sub in np.unique(subclasses):
        mask = subclasses == sub
        y_true = labels[mask]
        y_pred = preds[mask]
        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        tn = int(((y_pred == 0) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())
        prec  = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
        rec   = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        spec  = tn / (tn + fp) if (tn + fp) > 0 else float("nan")  # for Normal classes
        f1    = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else float("nan")
        rows.append({
            "subclass": sub,
            "binary_label": int(BINARY_LABEL.get(sub, -1)),
            "n": int(mask.sum()),
            "precision": round(prec, 3),
            "recall": round(rec, 3),
            "specificity": round(spec, 3),
            "f1": round(f1, 3),
            "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        })
    return pd.DataFrame(rows).sort_values(["binary_label", "subclass"])


label_tables = {}
for cfg in CONFIG_NAMES:
    all_logits, all_labels, all_subs = [], [], []
    for fold, run_dir in all_runs[cfg].items():
        preds = load_preds(run_dir)
        if preds is None:
            continue
        logits, labels, subclasses = preds
        all_logits.append(logits)
        all_labels.append(labels)
        all_subs.append(subclasses)
    if all_logits:
        t = per_label_metrics(
            np.concatenate(all_logits),
            np.concatenate(all_labels),
            np.concatenate(all_subs),
        )
        label_tables[cfg] = t
        print(f"\n── {DISPLAY_NAMES[cfg]} ──")
        display(t.set_index("subclass")[["binary_label", "n", "precision", "recall", "f1", "specificity"]])

if not label_tables:
    print("No val_preds_best.npz files found yet.")

In [ ]:
# Recall heatmap across all models — one row per subclass, one col per model
# Uses recall for suspicious classes (binary_label=1) and specificity for normal (binary_label=0)
if label_tables:
    first_cfg = next(iter(label_tables))
    t_first = label_tables[first_cfg]
    subclasses_all = t_first["subclass"].tolist()
    binary_label_map = dict(zip(t_first["subclass"], t_first["binary_label"]))

    display_matrix = np.full((len(subclasses_all), len(label_tables)), float("nan"))
    model_names = []

    for col_idx, (cfg, t) in enumerate(label_tables.items()):
        model_names.append(DISPLAY_NAMES[cfg])
        for row_idx, sub in enumerate(subclasses_all):
            row = t[t["subclass"] == sub]
            if not row.empty:
                bl = row.iloc[0]["binary_label"]
                display_matrix[row_idx, col_idx] = (
                    row.iloc[0]["recall"] if bl == 1 else row.iloc[0]["specificity"]
                )

    fig, ax = plt.subplots(figsize=(max(5, len(model_names) * 2.5), max(4, len(subclasses_all) * 0.55)))
    im = ax.imshow(display_matrix, aspect="auto", vmin=0, vmax=1, cmap="RdYlGn")
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=20, ha="right", fontsize=9)
    ax.set_yticks(range(len(subclasses_all)))
    ax.set_yticklabels(subclasses_all, fontsize=8)

    for i in range(len(subclasses_all)):
        for j in range(len(model_names)):
            v = display_matrix[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        fontsize=8, color="black" if 0.25 < v < 0.75 else "white")

    plt.colorbar(im, ax=ax, label="Recall (suspicious) / Specificity (normal)")
    ax.set_title("Per-original-label recall/specificity by model (both folds combined)")
    fig.tight_layout()
    plt.show()

## 8 · Optimal threshold analysis

In [ ]:
if label_tables:
    fig, axes = plt.subplots(1, len(label_tables), figsize=(5 * len(label_tables), 4), sharey=True)
    if len(label_tables) == 1:
        axes = [axes]

    for ax, (cfg, _) in zip(axes, label_tables.items()):
        all_logits, all_labels = [], []
        for fold, run_dir in all_runs[cfg].items():
            preds = load_preds(run_dir)
            if preds is None:
                continue
            logits, labels, _ = preds
            all_logits.append(logits)
            all_labels.append(labels)

        if not all_logits:
            continue

        logits = np.concatenate(all_logits)
        labels = np.concatenate(all_labels)
        probs  = 1 / (1 + np.exp(-logits))

        thresholds = np.linspace(0.1, 0.9, 80)
        recalls, specificities, f1s = [], [], []
        for t in thresholds:
            p = (probs >= t).astype(int)
            tp = ((p == 1) & (labels == 1)).sum()
            fp = ((p == 1) & (labels == 0)).sum()
            tn = ((p == 0) & (labels == 0)).sum()
            fn = ((p == 0) & (labels == 1)).sum()
            rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
            spec = tn / (tn + fp) if (tn + fp) > 0 else 0
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0
            f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
            recalls.append(rec)
            specificities.append(spec)
            f1s.append(f1)

        ax.plot(thresholds, recalls,       label="Recall (Suspicious)", color="#e53935")
        ax.plot(thresholds, specificities, label="Specificity (Normal)", color="#1e88e5")
        ax.plot(thresholds, f1s,           label="F1",                  color="#43a047")
        ax.axvline(0.5, color="grey", ls=":", lw=1)
        ax.set_xlabel("Threshold")
        ax.set_ylabel("Score")
        ax.set_title(DISPLAY_NAMES[cfg], fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    fig.suptitle("Recall / Specificity / F1 vs decision threshold", fontsize=11)
    fig.tight_layout()
    plt.show()
else:
    print("No prediction data available yet.")

## 9 · Phase 7 diagnosis

In [ ]:
sys.path.insert(0, str(Path("..") / "src"))
from kvasir_triage.eval.diagnose import classify_path, PR_AUC_THRESHOLD, NPV_THRESHOLD

if not summary_df.empty:
    print(f"Phase 7 thresholds: PR-AUC ≥ {PR_AUC_THRESHOLD}  &  NPV ≥ {NPV_THRESHOLD}\n")
    for cfg in CONFIG_NAMES:
        rows_cfg = summary_df[summary_df["model"] == DISPLAY_NAMES[cfg]]
        if rows_cfg.empty:
            continue
        mean_prauc = rows_cfg["pr_auc"].mean()
        mean_npv   = rows_cfg["npv"].mean()
        cv_metrics = {"aggregate": {
            "pr_auc": {"mean": mean_prauc},
            "npv":    {"mean": mean_npv},
        }}
        path = classify_path(cv_metrics)
        path_desc = {"A": "SUCCESS", "B": "Clever Hans", "C": "data-bound / underfit"}
        status = "✓" if path == "A" else "✗"
        print(f"  {status}  {DISPLAY_NAMES[cfg]:<30}  "
              f"PR-AUC={mean_prauc:.3f}  NPV={mean_npv:.3f}  →  Path {path} ({path_desc[path]})")
else:
    print("No completed runs to diagnose yet.")